<a href="https://colab.research.google.com/github/JeonghanSeo/KU-scRNAseq-seminar/blob/main/notebooks/01_scRNAseq_full.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# scRNA-seq 분석 실습

건국대학교 이형우 교수님 연구실 온라인 세미나
Data: **GSE210543** — Human fetal & adult **RPE-choroid** 조직 (Young vs Old, AMD 연구)
Runtime: **Google Colab + Seurat v5 (R)**

> ℹ️ 이 데이터는 **신경망막(neuroretina)이 아니라 RPE(망막색소상피)–맥락막(choroid)** 조직입니다. 자세한 내용은 Step 12를 참고하세요.

<table width="100%" cellpadding="0" cellspacing="0" style="background: #FFF1F2; border: 2px solid #E11D48; border-radius: 10px; margin: 12px 0;">
<tr><td style="padding: 14px 20px;">
<span style="font-size: 18px; font-weight: 700; color: #881337;">⚠️ 세미나 진행 중 주의사항 — 시작 전 꼭 읽어주세요</span><br/>
<span style="font-size: 13px; color: #9F1239;">세션이 끊기면 패키지 재설치에 20~30분이 걸립니다. 아래를 지켜주세요.</span>
</td></tr>
</table>

### 🔴 절대 금지 — 하면 세션이 죽어 전부 날아갑니다 (재설치 20~30분)
- **노트북 덮기 / 화면 잠금 / 절전 모드** ← 가장 흔한 사고! **미리 절전 꺼두기**
- **브라우저·탭 닫기**, **자리 비우고 90분+ 방치**
- **런타임 유형 변경**(CPU→GPU 등), **런타임 초기화 / 삭제**
- **다른 Colab 노트북 실행** (동시 세션 제한으로 이 세션이 밀려남)
- **Wi-Fi 끊김 / 네트워크 전환 / VPN 토글**, **Google 계정 로그아웃**

### 🟡 주의
- **F5·새로고침** — 셀 실행 중엔 금지 (보통 재연결되지만 굳이 할 필요 없음)
- **"런타임 재시작"** — 메모리 객체 날아감 · **"모두 실행"** 함부로 누르지 않기
- **셀은 위 → 아래 순서대로** 실행, 코드 셀 실수로 수정하지 않기

### 🟢 해도 괜찮음
- **실행 "중단(Interrupt)"** — 함수만 멈추고 메모리·객체 유지, 안전
- 스크롤 / 셀 클릭 / 목차 이동 · 짧은 재연결(런타임 살아있으면 객체 유지)

> **💡 끊기면:** 런타임 재연결 → **Step 0 실행**("설치 스킵" 뜨면 재설치 불필요) → Step 1부터 다시.
>
> **핵심 3줄:** ① 절전 끄고 자리 지키기  ② 런타임 유형·다른 Colab 건드리지 않기  ③ 셀은 순서대로

## 목차

> 💡 Colab에서 보고 있다면 왼쪽 사이드바의 **목차(≡) 아이콘**을 클릭하면 아래 각 Step으로 바로 이동할 수 있어요. 이 표는 전체 흐름을 한눈에 보기 위한 요약입니다.

| Step | 내용 | 예상 시간 |
|------|------|:----:|
| 0. 데이터셋 개요 | GSE210543, Web Summary 읽는 법 (좋은 샘플 vs 나쁜 샘플) | — |
| Step 0 | 환경 설정 (패키지 1회 설치) | 20~30분 |
| Step 1 | 10X 데이터 불러오기 | 10분 |
| Step 2 | QC 이론 + 시각화 *(+ DoubletFinder 원리, Fig. 8)* | 30분 |
| Step 3 | QC 필터링 | 10분 |
| Step 4 | 정규화 — LogNormalize | 10분 |
| Step 5 | HVG 선택 | 5분 |
| — | **중간 체크포인트 저장** | — |
| Step 6 | Cell Cycle Scoring *(optional)* | 5분 |
| Step 7 | Scaling + PCA | 10분 |
| Step 8 | Integration — Harmony | 10분 |
| Step 9 | UMAP | 10분 |
| Step 10 | Clustering (Resolution sweep) | 15분 |
| Step 11 | FindAllMarkers | 15분 |
| Step 12 | Cell Type Annotation | 20분 |

전체 슬라이드/다이어그램은 **Fig. 0~15** 순서로 번호가 매겨져 있습니다 (Step 2에 DoubletFinder 다이어그램 Fig. 8이 포함되어 있어 Fig 번호가 슬라이드 순서와 살짝 다릅니다).

## 0. 데이터셋 개요 — GSE210543

### Cell Ranger Web Summary

10X Genomics Cell Ranger가 시퀀싱 완료 후 생성하는 **QC 리포트**입니다.
분석 시작 전 반드시 확인해야 하는 **3가지 지표**:

| 지표 | 의미 | 권장 기준 |
|------|------|---------|
| **Estimated Number of Cells** | Cell Ranger 추정 세포 수 | **> 2,000** |
| **Median Genes per Cell** | 세포당 검출 유전자 중앙값 | **> 1,500** |
| **Fraction Reads in Cells** | 세포에 할당된 reads 비율 | **> 70%** |

> **💡 Tip:** Mean Reads per Cell이 매우 높다면 세포 수가 너무 적어서 생기는 **수학적 효과**일 수 있습니다 — 단독으로 해석하지 마세요.

### 전체 13개 샘플 품질 요약

| 샘플 | 그룹 | 세포 수 | Mean Reads/Cell | Median Genes/Cell | 선택 |
|------|------|-------:|---------------:|------------------:|:----:|
| 16PCW | Young | 8,601 | 43,243 | 1,084 | — |
| **20PCW** | Young | **5,787** | 103,036 | **5,174** | ✅ |
| **12PCW** | Young | **3,637** | 164,166 | **4,596** | ✅ |
| 21PCW | Young | 3,948 | 139,139 | 1,953 | — |
| **Adult_2** | Old | **6,516** | 90,897 | 2,016 | ✅ |
| **Adult_3** | Old | **3,694** | 142,303 | 2,806 | ✅ |
| Adult_5 | Old | 3,487 | 168,631 | 2,529 | — |
| Adult_1 | Old | 884 | 412,206 | 2,807 | ❌ |
| Adult_4 | Old | 496 | 1,146,570 | 2,414 | ❌ |
| AMD_macula | AMD | 1,719 | 284,749 | 4,002 | — |
| AMD_peripheral | AMD | 1,794 | 267,813 | 3,584 | — |
| Unaffected_macula | Control | 3,557 | 101,143 | 5,102 | — |
| Unaffected_peripheral | Control | 3,527 | 112,731 | 4,765 | — |

✅ 세미나 분석 샘플 (Young 2 + Old 2) | ❌ 세포 수 부족 또는 품질 이슈

### Web Summary 읽는 법 — 좋은 샘플 vs 나쁜 샘플

**✅ 좋은 샘플 — 20PCW**

<table width="100%" cellpadding="0" cellspacing="0" style="background: #F0FDF4; border: 2px solid #16A34A; border-radius: 10px; margin: 18px 0 4px 0;">
<tr>
  <td style="padding: 14px 20px; vertical-align: middle;">
    <span style="background: #16A34A; color: white; padding: 3px 12px; border-radius: 12px; font-size: 12px; font-weight: bold; margin-right: 10px;">GOOD</span>
    <span style="font-size: 17px; font-weight: 700; color: #14532D;">20PCW — 이상적인 라이브러리</span>
  </td>
  <td align="right" style="padding: 14px 20px; color: #166534; font-size: 13px; white-space: nowrap; vertical-align: middle;">5,787 cells &middot; 5,174 genes/cell &middot; 91.3% in cells</td>
</tr>
</table>

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/websummary_good_20PCW.png" width="900"/>

| 항목 | 값 | 평가 |
|------|-----|------|
| Estimated Cells | **5,787** | ✅ 충분 |
| Mean Reads/Cell | 103,036 | ✅ 정상 범위 |
| Median Genes/Cell | **5,174** | ✅ 매우 우수 |
| Fraction in Cells | **91.3%** | ✅ 배경 노이즈 최소 |

Barcode Rank Plot에서 파란 선과 회색 선 사이의 **꺾임(knee)**이 뚜렷합니다.

---

**❌ 나쁜 샘플 — Adult_4**

<table width="100%" cellpadding="0" cellspacing="0" style="background: #FFF1F2; border: 2px solid #E11D48; border-radius: 10px; margin: 18px 0 4px 0;">
<tr>
  <td style="padding: 14px 20px; vertical-align: middle;">
    <span style="background: #E11D48; color: white; padding: 3px 12px; border-radius: 12px; font-size: 12px; font-weight: bold; margin-right: 10px;">BAD</span>
    <span style="font-size: 17px; font-weight: 700; color: #881337;">Adult_4 — 세포 포획 실패</span>
  </td>
  <td align="right" style="padding: 14px 20px; color: #BE123C; font-size: 13px; white-space: nowrap; vertical-align: middle;">496 cells &middot; 1,146,570 reads/cell</td>
</tr>
</table>

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/websummary_bad_Adult4.png" width="900"/>

| 항목 | 값 | 평가 |
|------|-----|------|
| Estimated Cells | **496** | ❌ 세포 포획 실패 |
| Mean Reads/Cell | **1,146,570** | ⚠️ 비정상 (수학적 효과) |
| Median Genes/Cell | 2,414 | ⚠️ 보통 |
| Fraction in Cells | 78.8% | ⚠️ 낮은 편 |

```
Mean Reads/Cell = 총 reads ÷ 세포 수
               = 568,698,782 ÷ 496 ≈ 1,146,570
```

→ 총 reads는 20PCW(596M)와 비슷하지만 세포 수가 극히 적어 값이 폭등.
**세포 포획 실패**가 원인 — 분석에서 제외.

## Step 0. 환경 설정

> **Colab 설정:** Runtime → Change runtime type → **R**

**먼저 아래 설치 셀부터 실행하세요** (약 20~30분 소요). 설치되는 동안 이어지는 슬라이드로 개요를 보시면 됩니다 — 기다리는 시간 없이 진행할 수 있어요.

In [ ]:
# 패키지 설치 (약 20~30분, 세션마다 1회) — Drive 불필요, 각자 실행합니다
#
# 바이너리 저장소 사용: 소스 컴파일 대신 미리 빌드된 바이너리를 받아서
# 컴파일 실패(메모리 부족, 라이브러리 버전 문제 등)를 피하고 훨씬 빠르게 설치합니다.
# Ubuntu 코드네임을 자동 감지해서 Colab 이미지가 바뀌어도 안전하게 동작합니다.
ubuntu_codename <- system("lsb_release -cs", intern = TRUE)
repo_url <- sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", ubuntu_codename)
cat("바이너리 저장소:", repo_url, "

")

# 이미 이번 세션에 설치되어 있으면 건너뛰기 (셀 재실행 대비)
# showtext: 그래프 안의 한글이 깨지는 문제를 고치기 위해 추가 (Colab 기본 폰트는 한글 미지원)
required_pkgs <- c("Seurat", "harmony", "ggplot2", "patchwork", "showtext", "dplyr")
already_installed <- rownames(installed.packages())
missing_pkgs <- setdiff(required_pkgs, already_installed)

if (length(missing_pkgs) > 0) {
  cat("설치 필요:", paste(missing_pkgs, collapse = ", "), "

")
  install.packages(missing_pkgs, repos = repo_url)
} else {
  cat("이미 모두 설치되어 있습니다 — 설치 스킵
")
}

# 바로 로드까지 완료
library(Seurat)
library(harmony)
library(ggplot2)
library(patchwork)
library(showtext)
library(dplyr)

# ── 한글 폰트 설정 ──────────────────────────────────────────────
# 나눔고딕 설치 후 showtext로 등록 — 이후 모든 ggplot 기반 플롯
# (VlnPlot, DimPlot, ElbowPlot 등)의 한글 제목/텍스트가 정상 표시됩니다.
system("apt-get -qq update && apt-get -qq install -y fonts-nanum > /dev/null 2>&1")
font_add(family = "nanum", regular = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
showtext_auto()
showtext_opts(dpi = 96)
theme_set(theme_gray(base_size = 11, base_family = "nanum"))

# ── 플롯 출력 크기 ────────────────────────────────────────────
# Colab R 기본값(약 7x7 inch)이 좁아서 다중 패널 플롯이 답답하게 나옴 —
# 더 넓게 + 해상도도 올려서 전체 노트북 플롯에 적용
options(repr.plot.width = 12, repr.plot.height = 6, repr.plot.res = 150)

# ── presto (선택): FindAllMarkers 속도·메모리 대폭 개선 ──────
# CRAN에 없어 GitHub 소스 설치. 실패해도 그냥 넘어감(설치를 절대 안 깨뜨림).
tryCatch({
  if (!requireNamespace("presto", quietly = TRUE)) {
    if (!requireNamespace("remotes", quietly = TRUE))
      install.packages("remotes", repos = repo_url)
    remotes::install_github("immunogenomics/presto", upgrade = "never")
  }
  cat("presto 준비 완료 — FindAllMarkers 가속됨
")
}, error = function(e) cat("presto 설치 스킵 (기본 방식으로 진행)
"))

set.seed(42)
cat("
준비 완료!
")
cat("Seurat:", as.character(packageVersion("Seurat")), "
")

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_04.png" width="850"/>

*Fig. 0 — scRNA-seq 분석의 복잡성 (Hicks et al., BioRxiv 2015)*

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_05.png" width="850"/>

*Fig. 1 — scRNA-seq 전체 분석 워크플로우 (Luecken & Theis, Mol Syst Biol 2019)*

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_06.png" width="850"/>

*Fig. 2 — Seurat 기반 분석 워크플로우*

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_07.png" width="850"/>

*Fig. 3 — Seurat vs Scanpy 비교 및 Tip 1*

## Step 1. 데이터 불러오기

10X Genomics `filtered_feature_bc_matrix` 폴더를 읽어 **Seurat 오브젝트**를 생성합니다.

```
filtered_feature_bc_matrix/
├── barcodes.tsv.gz   ← 세포 바코드
├── features.tsv.gz   ← 유전자 ID / 이름
└── matrix.mtx.gz     ← UMI count matrix (sparse)
```

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_08.png" width="850"/>

*Fig. 4 — Count Matrix 생성 원리: barcodes / features / matrix (Macosko et al., Cell 2015)*

In [ ]:
# GitHub에서 데이터 가져오기 (Drive 불필요!)
# 저장소에 13개 샘플의 filtered_feature_bc_matrix가 모두 올라가 있습니다.
REPO_DIR <- "/content/KU-scRNAseq-seminar"
DATA_DIR <- file.path(REPO_DIR, "data/filtered_feature_bc_matrix")

if (!dir.exists(DATA_DIR)) {
  cat("GitHub에서 데이터 다운로드 중 (약 1~2분 소요)...
")
  ret <- system(paste0(
    "git clone --depth 1 https://github.com/JeonghanSeo/KU-scRNAseq-seminar.git ",
    REPO_DIR
  ))
  if (ret != 0) {
    stop("GitHub 클론 실패 — 네트워크 연결을 확인하세요")
  }
  cat("완료!
")
} else {
  cat("이미 다운로드됨 — 스킵
")
}

cat("
데이터 경로:", DATA_DIR, "
")
cat("샘플 목록:
")
samples_found <- list.dirs(DATA_DIR, recursive = FALSE, full.names = FALSE)
cat(paste(" ", samples_found, collapse = "
"), "
")

In [ ]:
# 4개 샘플을 하나씩 로드합니다
# Read10X: 10X Genomics 출력 파일(barcodes, features, matrix)을 읽어옵니다

# ── Young 샘플 ──────────────────────────────────────────────────
counts <- Read10X(data.dir = file.path(DATA_DIR, "12PCW"))
s_Young_12PCW <- CreateSeuratObject(counts = counts, project = "Young_12PCW",
                                    min.cells = 3, min.features = 200)
s_Young_12PCW$sample <- "Young_12PCW"
s_Young_12PCW$group  <- "Young"

counts <- Read10X(data.dir = file.path(DATA_DIR, "20PCW"))
s_Young_20PCW <- CreateSeuratObject(counts = counts, project = "Young_20PCW",
                                    min.cells = 3, min.features = 200)
s_Young_20PCW$sample <- "Young_20PCW"
s_Young_20PCW$group  <- "Young"

# ── Old 샘플 ────────────────────────────────────────────────────
counts <- Read10X(data.dir = file.path(DATA_DIR, "Adult_2"))
s_Old_Adult2 <- CreateSeuratObject(counts = counts, project = "Old_Adult2",
                                   min.cells = 3, min.features = 200)
s_Old_Adult2$sample <- "Old_Adult2"
s_Old_Adult2$group  <- "Old"

counts <- Read10X(data.dir = file.path(DATA_DIR, "Adult_3"))
s_Old_Adult3 <- CreateSeuratObject(counts = counts, project = "Old_Adult3",
                                   min.cells = 3, min.features = 200)
s_Old_Adult3$sample <- "Old_Adult3"
s_Old_Adult3$group  <- "Old"

# 목록으로 묶기
seurat_list <- list(
  Young_12PCW = s_Young_12PCW,
  Young_20PCW = s_Young_20PCW,
  Old_Adult2  = s_Old_Adult2,
  Old_Adult3  = s_Old_Adult3
)

# 결과 확인
cat("샘플별 세포 수:\n")
cat("  Young_12PCW:", ncol(s_Young_12PCW), "cells\n")
cat("  Young_20PCW:", ncol(s_Young_20PCW), "cells\n")
cat("  Old_Adult2 :", ncol(s_Old_Adult2),  "cells\n")
cat("  Old_Adult3 :", ncol(s_Old_Adult3),  "cells\n")


**▶ 결과 해석**

각 샘플의 세포 수를 확인하세요:

- `CreateSeuratObject(min.cells=3)` — 최소 3개 세포에서 발현된 유전자만 포함
- `min.features=200` — 200개 미만 유전자 발현 세포는 이 단계에서 제거됨
- Cell Ranger 추정치보다 낮을 수 있음 (엄격한 초기 필터링 효과)

> **Young 샘플** (12PCW, 20PCW): 발달기 세포, 증식 세포 혼재 가능
> **Old 샘플** (Adult_2, Adult_3): 성체 성숙 세포, 구성이 안정적

## Step 2. QC — Quality Control

### 핵심 QC 지표 3가지

| 지표 | 의미 | 기본 필터 |
|------|------|---------|
| `nFeature_RNA` | 세포당 검출 유전자 수 | **> 200** |
| `nCount_RNA` | 세포당 총 UMI 수 | **> 500** |
| `percent.mt` | 미토콘드리아 유전자 비율 | **< 10%** |

**낮은 nFeature** → 빈 droplet 또는 죽은 세포
**높은 percent.mt** → 세포막 손상, 세포질 RNA 유출

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_09.png" width="850"/>

*Fig. 5 — QC 지표 분포 확인 방법 (NCells, nUMI, nGene, mitoRatio)*

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_10.png" width="850"/>

*Fig. 6 — QC VlnPlot 예시: nFeature_RNA / nCount_RNA / percent.mt*

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_11.png" width="850"/>

*Fig. 7 — QC 필터링 실습 코드 및 Tip 2: Filter 조건에서의 정답은 없다!*

In [ ]:
# 미토콘드리아 유전자 비율 계산
# 인간 미토콘드리아 유전자는 이름이 'MT-' 로 시작합니다
# 높은 비율 = 세포막이 손상되어 세포질 mRNA가 빠져나간 상태 (죽은 세포)

seurat_list[["Young_12PCW"]][["percent.mt"]] <- PercentageFeatureSet(seurat_list[["Young_12PCW"]], pattern = "^MT-")
seurat_list[["Young_20PCW"]][["percent.mt"]] <- PercentageFeatureSet(seurat_list[["Young_20PCW"]], pattern = "^MT-")
seurat_list[["Old_Adult2"]][["percent.mt"]]  <- PercentageFeatureSet(seurat_list[["Old_Adult2"]],  pattern = "^MT-")
seurat_list[["Old_Adult3"]][["percent.mt"]]  <- PercentageFeatureSet(seurat_list[["Old_Adult3"]],  pattern = "^MT-")

# QC 지표 확인 (첫 6개 세포 예시)
head(seurat_list[["Young_12PCW"]]@meta.data[, c("nFeature_RNA", "nCount_RNA", "percent.mt")])


In [ ]:
# QC 분포 시각화
# 4개 샘플을 임시로 합쳐서 한 번에 비교합니다

seurat_qc_merged <- merge(
  seurat_list[["Young_12PCW"]],
  y = list(seurat_list[["Young_20PCW"]],
           seurat_list[["Old_Adult2"]],
           seurat_list[["Old_Adult3"]]),
  add.cell.ids = names(seurat_list)
)
seurat_qc_merged[["percent.mt"]] <- PercentageFeatureSet(seurat_qc_merged, pattern = "^MT-")

VlnPlot(
  seurat_qc_merged,
  features = c("nFeature_RNA", "nCount_RNA", "percent.mt"),
  group.by = "sample",
  ncol     = 3,
  pt.size  = 0
) & theme(axis.text.x = element_text(angle = 45, hjust = 1))


In [ ]:
# QC 통계 요약 — 샘플별로 확인합니다
# 이 숫자를 보고 아래 필터링 임계값을 결정하세요

for (name in names(seurat_list)) {
  obj <- seurat_list[[name]]
  cat("──", name, "──────────────────────\n")
  cat("  세포 수         :", ncol(obj), "\n")
  cat("  nFeature 중앙값 :", round(median(obj$nFeature_RNA)), "\n")
  cat("  nFeature 95%ile :", round(quantile(obj$nFeature_RNA, 0.95)), "\n")
  cat("  nCount 중앙값   :", round(median(obj$nCount_RNA)), "\n")
  cat("  nCount 98%ile   :", round(quantile(obj$nCount_RNA, 0.98)), " ← Singlet 기준\n")
  cat("  MT% 중앙값      :", round(median(obj$percent.mt), 2), "\n")
  cat("\n")
}


**▶ 결과 해석**

VlnPlot과 통계에서 확인할 항목:

| 지표 | 낮은 값 문제 | 높은 값 문제 |
|------|------------|------------|
| `nFeature_RNA` | 빈 droplet / 죽은 세포 | **Doublet** (두 세포가 하나로 포획) |
| `nCount_RNA` | 낮은 품질 | Doublet 의심 |
| `percent.mt` | — | 세포막 손상 |

**nFeature 상한선 설정 기준**: 샘플의 **95th percentile** 또는 분포에서 뚜렷한 어깨(shoulder)가 보이는 지점

> **고급 QC — DoubletFinder 원리**
> nFeature/nCount 상한선은 doublet을 대략적으로만 걸러냅니다. 이 세미나에서는 시간 제약으로
> 실제 실행은 생략하지만(샘플당 10~20분), **원리를 알아두면 QC 결과를 해석하는 데 도움이 됩니다.**
>
> `DoubletFinder`(McGinnis, Murrow & Gartner, *Cell Systems* 2019, [doi:10.1016/j.cels.2019.03.003](https://doi.org/10.1016/j.cels.2019.03.003))는
> **"인공 doublet(artificial doublet)"을 만들어 실제 세포와 비교**하는 방식으로 작동합니다:
>
> 1. **인공 doublet 생성** — 실제 세포 중 무작위로 2개를 골라 발현 프로파일을 평균내어 "가짜 doublet"을 대량 생성 (전체 세포 수의 약 25%, `pN` 파라미터)
> 2. **함께 임베딩** — 인공 doublet을 실제 데이터와 합쳐서 PCA를 다시 수행 → 같은 저차원 공간에 배치
> 3. **이웃 비율 계산 (pANN)** — 실제 세포마다 k개의 최근접 이웃 중 인공 doublet이 차지하는 비율을 계산. 진짜 doublet은 발현 프로파일이 "평균적"이라 인공 doublet 근처에 몰리는 경향이 있음
> 4. **임계값으로 분류** — pANN이 높은 상위 세포들을 doublet으로 판정. 몇 개를 doublet으로 볼지는 **예상 doublet 비율(expected multiplet rate)**로 결정
>
> 즉, "발현 패턴이 두 세포를 섞어놓은 것과 비슷한가?"를 인공적으로 만든 정답(doublet)과 비교해 판단하는 방식입니다. Singlet QC(nCount percentile)는 이 판단을 **크기(UMI 총량)** 하나로 근사한 것이라고 볼 수 있습니다.

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/doubletfinder_principle.svg" width="900"/>

*Fig. 8 — DoubletFinder의 3단계 원리 (McGinnis et al. 2019의 개념을 참고해 재구성한 다이어그램). 실제 세포(파란 원)와 인공 doublet(빨간 마름모)을 함께 PCA 공간에 배치한 뒤, 각 세포의 k-최근접 이웃 중 인공 doublet 비율(pANN)을 계산합니다. pANN이 threshold를 넘는 세포가 "Predicted Doublets"로 분류됩니다.*

> **① 인공 doublet 생성이 왜 필요한가?** 진짜 doublet의 "정답표"는 없습니다 (두 세포가 섞인 걸 되돌릴 방법이 없으니까요). 그래서 우리가 직접 정답을 만듭니다 — 실제 세포 중 무작위 쌍을 골라 평균내면, "진짜 doublet과 통계적으로 유사한" 인공 데이터를 얻을 수 있습니다.
>
> **② 왜 PCA 공간에서 판단하나?** 원본 유전자 발현 공간은 차원이 너무 높고 노이즈가 많습니다. PCA로 축소하면 "세포 타입이 비슷한 정도"가 거리로 표현되고, 인공 doublet은 자신이 유래한 두 세포 타입의 "중간 지점"에 위치하는 경향이 뚜렷해집니다.
>
> **③ pANN이 왜 doublet 신호가 되나?** 진짜 doublet도 인공 doublet과 마찬가지로 "두 세포 타입의 중간 발현 프로파일"을 가지므로, PCA 공간에서 인공 doublet들과 가까운 이웃이 됩니다. 즉 **주변 이웃 중 인공 doublet 비율이 높다 = 이 세포도 두 세포가 섞였을 가능성이 높다**는 논리입니다.

**GEM-X 화학(chemistry)에서는 뭐가 달라지나?**

10x Genomics는 2023년 이후 기존 **Next GEM** 화학을 **GEM-X**로 업그레이드했습니다. 아래는 [10x Genomics GEM-X 기술 블로그](https://www.10xgenomics.com/blog/the-next-generation-of-single-cell-rna-seq-an-introduction-to-gem-x-technology)에 실제로 인용된 수치만 정리한 비교표입니다 (확인되지 않은 수치는 제외했습니다):

| 항목 | Next GEM (기존) | GEM-X (최신) | 비고 |
|---|---:|---:|---|
| Multiplet rate | 회수 세포 1,000개당 ≈ 0.8%p | 회수 세포 1,000개당 ≈ 0.4%p | 2-fold 감소, HEK293T/NIH-3T3 mixing 실험 (10x 블로그 Fig. 9) |
| 레인당 세포 처리량 | 최대 ~10,000–15,000 | 최대 ~20,000 | 10x 블로그 Fig. 9 |
| 세포 회수 효율 — 3′ GEX | 60% | 75% | 10x 블로그 Fig. 7 |
| 세포 회수 효율 — 5′ GEX | 54% | 60% | 10x 블로그 Fig. 7 |
| 검출 감도 — 마우스 뇌 | 기준 | 유전자 +98%, 전사체(UMI) +249% | 10x 블로그 Fig. 6 |
| 검출 감도 — 인간 PBMC (5′) | 기준 | 유전자 +61%, 전사체(UMI) +103% | 10x 블로그 Fig. 8, 12 |
| 칩 partitioning 시간 | (원문에 비교 수치 없음) | 6분 | "세포 스트레스에 부정적 영향 없음"이라고 명시 |

**왜 이 수치들이 의미가 있나?**
- **회수 효율(recovery)**이 높을수록 같은 수의 세포를 로딩해도 더 많이 건져낼 수 있어서, 생검(biopsy)처럼 세포 수가 제한된 귀한 샘플에서 손실을 줄일 수 있습니다.
- **검출 감도**가 높다는 건 세포당 더 많은 유전자·UMI를 잡아낸다는 뜻이라, 전사인자(TF)나 사이토카인처럼 원래 발현량이 낮은 유전자를 놓치지 않을 확률이 올라갑니다.
- **Multiplet rate 감소**는 이번 절의 핵심 주제죠 — 같은 세포 수를 로딩해도 doublet이 덜 생기니, DoubletFinder 같은 사후 필터링에 걸리는 세포도 줄어듭니다.

> ⚠️ **우리 세미나 데이터(GSE210543)는 GEM-X 이전, 즉 Next GEM 화학으로 생성**됐습니다 — GEO 등록 정보상 2022년 8월 제출, 2023년 1월 공개로, GEM-X가 상용화되기 전입니다. 그래서 아래 코드는 **Next GEM 기준(1,000개당 0.8%p)**을 기본값으로 사용합니다. 여러분이 나중에 직접 만든 GEM-X 데이터를 분석할 때는 `chemistry = "gemx"`로 바꿔서 쓰면 됩니다.
>
> 이 표는 10x의 **일반 GEX(유전자 발현) 실험용** 가이드입니다. CRISPR 스크린에 쓰이는 GEM-X Universal 5' Singleplex의 "internal hashing" 표(sgRNA로 doublet을 사후에 걸러낼 수 있어 20%+ 까지도 허용)와는 다른 표이니 혼동하지 마세요.

바로 아래 코드 셀에서, 이 근사식을 우리 4개 샘플의 **실제 필터링된 세포 수**에 적용해 샘플별 예상 multiplet rate를 계산합니다.

In [ ]:
# nFeature vs nCount 산점도 — doublet 탐지
# 정상 세포: 두 지표가 선형 관계
# Doublet: nCount, nFeature 모두 비정상적으로 높음

# Young_12PCW 예시
p1 <- FeatureScatter(seurat_list[["Young_12PCW"]],
                     feature1 = "nCount_RNA",
                     feature2 = "percent.mt") +
      ggtitle("Young_12PCW - Count vs MT%")

p2 <- FeatureScatter(seurat_list[["Young_12PCW"]],
                     feature1 = "nCount_RNA",
                     feature2 = "nFeature_RNA") +
      ggtitle("Young_12PCW - Count vs Feature")

p1 + p2


**▶ 결과 해석**

**Count vs MT%**: 정상 세포는 낮은 MT% 유지. MT% 가 높은 세포 → 세포막 손상 → 제거 대상
**Count vs Feature**: 정상 세포는 **선형 관계**를 보입니다.

- 오른쪽 상단 이상치 → UMI는 많은데 유전자 수도 많음 → **doublet 의심**
- 왼쪽 하단 이상치 → 유전자/UMI 모두 낮음 → **빈 droplet** 의심
- 선형 관계에서 벗어난 점들이 QC 필터링 대상입니다

## Step 3. QC 필터링

```r
nFeature_RNA > 200  &  nCount_RNA > 500  &  percent.mt < 10
```

> **💡 Tip:** 임계값은 샘플마다 다릅니다. VlnPlot을 보고 분포의 자연스러운 경계(valley)를 찾으세요.

In [ ]:
# QC 기준값 설정
# ▶ 위 통계를 확인하고 필요하면 수정하세요!
MIN_GENE <- 200   # 최소 유전자 수   (적으면 빈 droplet)
MIN_UMI  <- 500   # 최소 UMI 수
MAX_MT   <- 10    # 최대 미토콘드리아 비율 (%)
# UMI 상위 2%는 doublet(두 세포가 한 droplet)으로 간주 → 샘플별로 계산

seurat_list_filtered <- list()   # 필터링 결과를 저장할 빈 목록

for (name in names(seurat_list)) {
  obj <- seurat_list[[name]]

  # 이 샘플의 nCount 98th percentile (doublet 상한선)
  max_umi <- quantile(obj$nCount_RNA, 0.98)

  # 조건에 맞는 세포만 남기기
  obj_filtered <- subset(obj,
    subset = nFeature_RNA > MIN_GENE &
             nCount_RNA   > MIN_UMI  &
             nCount_RNA   < max_umi  &   # 상위 2% 제거
             percent.mt   < MAX_MT)

  seurat_list_filtered[[name]] <- obj_filtered

  cat(name, ":", ncol(obj), "→", ncol(obj_filtered), "cells",
      "(제거:", ncol(obj) - ncol(obj_filtered), ")\n")
}


**▶ 결과 해석**

**3단계 QC 결과** 확인:

| 제거 비율 | 해석 |
|---------|------|
| **5~15%** | 정상적인 QC |
| **> 25%** | 임계값 너무 엄격하거나 샘플 품질 이슈 |
| **< 3%** | 임계값 너무 느슨할 수 있음 |

**nCount 98th percentile (Singlet QC):**
- 상위 2% 세포는 **doublet 또는 triplet** (두 세포 이상이 한 droplet에 포획)
- 샘플별로 다른 값 적용 — 데이터 기반 필터링
- 고정값(5000 등)보다 **합리적이고 재현성 높음**

> **DoubletFinder** 는 더 정밀하지만 샘플별 10~20분 소요
> 실제 분석에서는 Singlet QC → DoubletFinder 순서로 적용 권장

In [ ]:
# Multiplet rate 근사식 (10x Genomics 표준 가이드, 회수 세포 1,000개당 선형 근사)
# chemistry = "next_gem" (기본값, 0.8%p/1000) 또는 "gemx" (0.4%p/1000)
# 우리 세미나 데이터(GSE210543)는 2022년 8월 제출 / 2023년 1월 공개 —
# GEM-X가 상용화되기 전이라 "next_gem"이 정확한 값입니다.
# 여러분이 나중에 직접 만든 GEM-X 데이터를 쓸 때는 chemistry = "gemx"로 바꾸세요.
estimate_multiplet_rate <- function(n_cells_recovered, chemistry = "next_gem") {
  rate_per_1000 <- switch(chemistry,
    next_gem = 0.008,
    gemx     = 0.004,
    stop("chemistry는 'next_gem' 또는 'gemx' 중 하나여야 합니다")
  )
  rate_per_1000 * (n_cells_recovered / 1000)
}

# 우리 4개 샘플의 "실제 필터링된 세포 수" 기준으로 계산 (Next GEM 기준)
cat("샘플별 예상 multiplet rate (Next GEM 기준, GSE210543)
")
cat("──────────────────────────────────────────
")
for (name in names(seurat_list_filtered)) {
  n_cells <- ncol(seurat_list_filtered[[name]])
  rate    <- estimate_multiplet_rate(n_cells, chemistry = "next_gem")
  cat(sprintf("%-12s: %5d cells  →  ≈ %.1f%%
", name, n_cells, rate * 100))
}

# ── DoubletFinder 실제 실행 시 (참고용 — 이 세미나에서는 실행하지 않습니다) ──
# 공식 저장소: https://github.com/chris-mcginnis-ucsf/DoubletFinder
#
# 실제로 쓰게 되면 이 3곳만 우리 데이터에 맞게 손보면 됩니다:
#   ① exp_rate — 바로 위에서 계산한 "샘플별 예상 multiplet rate"를 그대로 재사용
#      (표를 다시 볼 필요 없이, 같은 estimate_multiplet_rate() 함수를 그대로 호출)
#   ② pK_optimal — paramSweep()으로 탐색 (샘플당 대부분의 시간이 여기서 소요됨)
#   ③ nExp — ①에서 나온 rate와 homotypic 보정을 곱해서 자동 계산됨
#
# for (name in names(seurat_list_filtered)) {
#   obj      <- seurat_list_filtered[[name]]
#   n_cells  <- ncol(obj)
#   exp_rate <- estimate_multiplet_rate(n_cells, chemistry = "next_gem")  # ① 위 표와 동일한 값
#
#   # 1) pK 최적값 탐색 (ground-truth 없이) — ②
#   sweep.res  <- paramSweep(obj, PCs = 1:30, sct = FALSE)
#   sweep.stat <- summarizeSweep(sweep.res, GT = FALSE)
#   bcmvn      <- find.pK(sweep.stat)
#   pK_optimal <- as.numeric(as.character(bcmvn$pK[which.max(bcmvn$BCmetric)]))
#
#   # 2) 예상 doublet 세포 수 (homotypic doublet 보정 포함) — ③
#   homotypic.prop <- modelHomotypic(obj$seurat_clusters)
#   nExp_poi       <- round(exp_rate * n_cells)
#   nExp_poi.adj   <- round(nExp_poi * (1 - homotypic.prop))
#
#   # 3) 실행
#   obj <- doubletFinder(obj, PCs = 1:30, pN = 0.25, pK = pK_optimal,
#                         nExp = nExp_poi.adj, sct = FALSE)
#   seurat_list_filtered[[name]] <- obj
# }

## Step 4. 정규화 — Normalization

세포마다 **시퀀싱 깊이(sequencing depth)**가 다릅니다. 정규화로 이를 보정합니다.

**LogNormalize** (기본값):
```
normalized = log(count / total_count × 10,000 + 1)
```
→ 총 UMI 수로 나눈 뒤 **log 변환** — 깊이 차이를 제거하고 분포를 안정화

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_12.png" width="850"/>

*Fig. 9 — LogNormalization vs SCTransform 방법 비교 및 PC 선택 기준 (Tip 3)*

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_13.png" width="850"/>

*Fig. 10 — 정규화 개념: SCTransform vs LogNormalization (M. Loven, RNA-seq statistical analysis)*

In [ ]:
# 정규화: 각 세포의 총 UMI가 10,000이 되도록 보정한 후 log 변환
# → 세포마다 포획된 mRNA 양의 차이를 제거합니다

for (name in names(seurat_list_filtered)) {
  seurat_list_filtered[[name]] <- NormalizeData(
    seurat_list_filtered[[name]],
    normalization.method = "LogNormalize",
    scale.factor         = 10000
  )
}
cat("정규화 완료!\n")


## Step 5. HVG — 고변이 유전자 선택

전체 수만 개 유전자 중 **샘플 간 차이가 큰 유전자**만 선택하여 분석 효율을 높입니다.

일반적으로 **상위 2,000개** 사용 (Seurat 기본값).
→ 모든 유전자를 쓰면 노이즈가 커지고 속도가 느려집니다.

In [ ]:
# 고변동 유전자(HVG) 선택: 샘플 간 발현 차이가 큰 유전자 2,000개
# → 세포 타입 구분에 정보가 많은 유전자들입니다

for (name in names(seurat_list_filtered)) {
  seurat_list_filtered[[name]] <- FindVariableFeatures(
    seurat_list_filtered[[name]],
    selection.method = "vst",
    nfeatures        = 2000
  )
}

# Top 10 HVG 확인 (12PCW 예시)
top10 <- head(VariableFeatures(seurat_list_filtered[["Young_12PCW"]]), 10)
cat("Top 10 고변동 유전자 (Young_12PCW):\n")
print(top10)


In [ ]:
# QC + 정규화 + HVG가 완료된 4개 샘플을 하나로 합칩니다

seurat_merged <- merge(
  x            = seurat_list_filtered[["Young_12PCW"]],
  y            = list(seurat_list_filtered[["Young_20PCW"]],
                      seurat_list_filtered[["Old_Adult2"]],
                      seurat_list_filtered[["Old_Adult3"]]),
  add.cell.ids = names(seurat_list_filtered)
)

# Seurat v5 필수: 샘플별로 분리된 레이어를 하나로 합칩니다
seurat_merged <- JoinLayers(seurat_merged)

# 합친 뒤 전체 데이터에서 HVG 재선택
seurat_merged <- FindVariableFeatures(seurat_merged, nfeatures = 2000)

cat("합병 완료!\n")
cat("전체 세포 수  :", ncol(seurat_merged), "\n")
cat("전체 유전자 수:", nrow(seurat_merged), "\n")
cat("\n샘플별 세포 수:\n")
print(table(seurat_merged$sample))

# ── 메모리 정리: 이후로는 seurat_merged만 사용 → 큰 중간 객체 제거 (OOM 방지) ──
rm(list = intersect(c("seurat_list", "seurat_list_filtered", "seurat_qc_merged"), ls()))
invisible(gc())


**▶ 결과 해석**

`merge()` + `JoinLayers()` 후 4개 샘플이 하나의 오브젝트로 통합됩니다.

- **JoinLayers()**: Seurat v5에서 샘플별로 분리된 data 레이어를 하나로 합침
- **FindVariableFeatures()** 재실행: 병합 후 전체 세포에서 다시 HVG 선택
- 샘플별 세포 수 확인 → 크게 불균형하면 integration 품질에 영향

---

## 중간 체크포인트

QC + 정규화 + HVG까지 완료했습니다. 오브젝트를 저장합니다.

> **💡 Tip:** 세션이 끊기면 Step 0 라이브러리 로드 후 아래 **로드 셀**을 실행하세요.

In [ ]:
# 저장 경로 설정 (처음 1회 실행)
SAVE_DIR <- "/content/drive/MyDrive/KU_seminar/results"
dir.create(SAVE_DIR, showWarnings = FALSE, recursive = TRUE)

# 오브젝트 저장
saveRDS(seurat_merged, file = file.path(SAVE_DIR, "01_seurat_merged.rds"))
cat("저장 완료:", file.path(SAVE_DIR, "01_seurat_merged.rds"), "\n")

In [ ]:
# ── 세션 재시작 시 여기서부터 ────────────────────────────────────
# (위 셀들을 다시 실행하지 않아도 됩니다)

# library(Seurat); library(harmony); library(dplyr); library(ggplot2); library(patchwork)
# set.seed(42)
# SAVE_DIR <- "/content/drive/MyDrive/KU_seminar/results"
# seurat_merged <- readRDS(file.path(SAVE_DIR, "01_seurat_merged.rds"))
# cat("로드 완료:", ncol(seurat_merged), "cells\n")

## Step 6. Cell Cycle Scoring *(Optional)*

세포 주기(**G1 / S / G2M**)가 클러스터링에 영향을 줄 수 있습니다.
**발달기(Young)** 샘플에는 증식 세포가 많으므로 확인이 필요합니다.

→ 주기 효과가 클 경우 `vars.to.regress`로 회귀 제거 가능

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_14.png" width="850"/>

*Fig. 11 — Cell Cycle Scoring: 언제 회귀할지, 언제 남겨둘지*

In [ ]:
# Cell cycle 관련 유전자 (Seurat 내장)
s.genes   <- cc.genes$s.genes    # S phase
g2m.genes <- cc.genes$g2m.genes  # G2M phase

# Cell cycle score 계산
seurat_merged <- CellCycleScoring(
  seurat_merged,
  s.features   = s.genes,
  g2m.features = g2m.genes,
  set.ident    = TRUE
)

# 분포 확인
table(seurat_merged$Phase)

## Step 7. Scaling + PCA

**Scaling**: 유전자별 **평균 0, 분산 1**로 맞춰 발현량 크기 차이를 제거합니다.
**PCA**: 고변이 유전자 2,000개 → **주요 성분 50개**로 차원 축소.

→ PCA ElbowPlot으로 유효한 PC 수를 확인하세요 (보통 **15~30개** 사용)

### PCA란? — 직관적으로 이해하기

PCA는 **"변이(variation)가 가장 큰 방향"을 찾아 데이터를 요약**하는 차원 축소 기법입니다.

**2D 비유 (유전자 2개로 생각해보기):**
- 두 유전자의 발현을 x, y축에 놓고 세포들을 뿌립니다
- 세포들이 **가장 넓게 퍼진 방향**으로 선을 그으면 → 그게 **PC1** (제1주성분, 변이 최대)
- PC1에 **수직**으로, 두 번째로 넓은 방향 → **PC2**
- 실제 데이터는 유전자가 2,000개(2,000차원)라, 이 방향을 **PC1~PC50**까지 찾습니다

**각 세포의 PC 점수는 어떻게 나오나?**
- 각 유전자는 PC마다 **"기여도(loading)"** 를 가집니다
- `세포의 PC1 점수 = Σ (유전자 발현 × 그 유전자의 PC1 기여도)` — 전체 유전자에 대해 합산
- 즉 **2,000개 유전자를 50개 PC 점수로 압축**해서 세포들을 비교하는 것입니다

**왜 하나?** 2,000개 유전자를 다 쓰면 노이즈가 크고 느립니다. 가장 큰 변이 원천(주로 **세포 타입 차이**)을 담은 **상위 PC 몇 개만** 이후 UMAP·클러스터링에 사용합니다.

> 📖 참고: [HBC scRNA-seq 튜토리얼 — Normalization & PCA](https://hbctraining.github.io/scRNA-seq/lessons/05_normalization_and_PCA.html)

In [ ]:
# Scaling: 유전자별 평균=0, 분산=1 정규화
# ── Cell cycle 회귀는 하지 않습니다 ──────────────────────────
#   Young(태아) vs Old(성체) 비교에서는 "증식(proliferation)"이 관심 있는
#   진짜 생물학적 차이라, 이를 regress로 제거하면 그 신호가 사라집니다.
#   (cell cycle을 순수 교란변수로 없애고 싶을 때만 아래 주석을 해제하세요.)
seurat_merged <- ScaleData(
  seurat_merged,
  # vars.to.regress = c("S.Score", "G2M.Score"),   # 필요시 주석 해제 (또는 CC.Difference = S.Score - G2M.Score)
  features        = VariableFeatures(seurat_merged)  # HVG만 scale (속도 ↑)
)

cat("Scaling complete!
")

In [ ]:
# PCA 실행
seurat_merged <- RunPCA(
  seurat_merged,
  features = VariableFeatures(seurat_merged),
  npcs     = 50
)

# Elbow Plot — 몇 개의 PC를 사용할지 결정
ElbowPlot(seurat_merged, ndims = 50) +
  ggtitle("Elbow Plot: PC 기여도") +
  geom_vline(xintercept = 30, linetype = "dashed", color = "red") +
  annotate("text", x = 32, y = 3, label = "PC=30 선택", color = "red")

**▶ 결과 해석**

**Elbow Plot**에서 표준편차가 급격히 감소하다가 완만해지는 **꺾임점(elbow)**을 찾으세요.

- 꺾임점 이후 PC들은 주로 **기술적 노이즈**를 반영합니다
- 이 데이터셋에서는 보통 **PC 20~30**이 적절합니다
- 의심스러우면 넉넉하게 (30) 선택 — 적게 쓰는 것보다 안전합니다

## Step 8. Integration — Harmony

**Young(발달기)**과 **Old(성체)** 샘플은 생물학적 차이 외에도 **배치 효과(batch effect)**가 있습니다.
**Harmony**로 배치를 보정하되 생물학적 신호는 보존합니다.

→ PCA space에서 샘플 레이블을 기준으로 반복 보정 수행

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_15.png" width="850"/>

*Fig. 12 — Integration 방법 비교: Harmony가 세포 타입 구조를 잘 보존 (Tran et al., Genome Biol 2020)*

### Harmony 통합 실행

`RunHarmony()` — PCA 공간에서 샘플별 분포를 반복적으로 보정합니다.

> **참고: Seurat v5의 두 가지 방법**
> - `RunHarmony()` ← 이 세미나 사용 (명확하고 빠름)
> - `IntegrateLayers(HarmonyIntegration)` — v5 네이티브, 결과 동일하지만 레이어 관리 필요

```
# SCTransform 방식 (더 정밀, 느림 — 고급)
# seurat_merged <- SCTransform(seurat_merged, vars.to.regress = c("S.Score","G2M.Score"))
# seurat_merged <- RunPCA(seurat_merged)
# seurat_merged <- RunHarmony(seurat_merged, group.by.vars = "sample")
```

In [ ]:
# Harmony 배치 보정
# group.by.vars: 보정할 배치 변수 (여기서는 샘플 ID)
seurat_merged <- RunHarmony(
  seurat_merged,
  group.by.vars    = "sample",   # 샘플별 배치 보정
  reduction        = "pca",      # PCA 결과를 입력으로 사용
  reduction.save   = "harmony",  # 결과 저장 위치
  max_iter         = 20          # 최대 반복 횟수 (보통 10회 내에 수렴) — 최신 harmony는 max_iter 사용
)

# 수렴 확인 — "Harmony converged after N iterations" 메시지 확인
cat("Harmony reduction dims:", ncol(Embeddings(seurat_merged, "harmony")), "\n")

**▶ 결과 해석**

Harmony 실행 후 확인할 내용:

- `converged after N iterations` — N이 작을수록 배치 효과가 뚜렷했음을 의미
- `max.iter` 도달 시 → `max.iter.harmony` 값을 늘려보세요
- 결과는 `seurat_merged@reductions$harmony` 에 저장됨 (PCA와 동일한 차원)

> **다음 단계**: 이 harmony embedding을 UMAP / Clustering 입력으로 사용합니다

## Step 9. UMAP

고차원 데이터를 **2D**로 시각화합니다.
Harmony 보정 결과(`reduction = "harmony"`)를 입력으로 사용합니다.

→ UMAP은 시각화 도구입니다. **클러스터링은 별도로 수행**하며 UMAP 좌표에 의존하지 않습니다.

In [ ]:
# UMAP (Harmony 통합 결과 기반)
seurat_merged <- RunUMAP(
  seurat_merged,
  reduction = "harmony",
  dims      = 1:30
)

# 통합 전/후 비교
p_before <- DimPlot(seurat_merged, reduction = "pca",  group.by = "sample") + ggtitle("Before Integration (PCA)")
p_after  <- DimPlot(seurat_merged, reduction = "umap", group.by = "sample") + ggtitle("After Integration (UMAP)")
p_before + p_after

**▶ 결과 해석**

| | Before Integration (PCA) | After Integration (UMAP) |
|--|--|--|
| 기대 패턴 | 샘플별로 분리됨 | 샘플이 섞여 하나의 구름 형성 |

- **Before**: Young/Old 샘플이 PCA에서 분리 → 배치 효과 확인
- **After**: Harmony 보정 후 같은 세포 타입끼리 모임 → 통합 성공

> UMAP에서 샘플이 여전히 분리된다면 → Harmony 파라미터 조정 필요

## Step 10. Clustering

**KNN 그래프** 기반 Louvain/Leiden 알고리즘으로 클러스터를 탐지합니다.
`resolution`이 **클수록 더 많은 클러스터**가 생성됩니다.

> **💡 Tip:** Resolution **0.4~0.6**에서 시작하세요. VlnPlot과 마커 유전자를 보고 최종 선택합니다.

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_16.png" width="850"/>

*Fig. 13 — Resolution에 따른 클러스터링 결과 비교 (Res 0.2 ~ 1.2)*

In [ ]:
# 이웃 그래프 생성 (Harmony 결과 기반)
seurat_merged <- FindNeighbors(
  seurat_merged,
  reduction = "harmony",
  dims      = 1:30
)

# 클러스터링 — resolution 숫자가 클수록 클러스터가 더 많이 나옵니다
seurat_merged <- FindClusters(seurat_merged, resolution = 0.2)
cat("Resolution 0.2 →", length(unique(seurat_merged$seurat_clusters)), "clusters\n")

seurat_merged <- FindClusters(seurat_merged, resolution = 0.4)
cat("Resolution 0.4 →", length(unique(seurat_merged$seurat_clusters)), "clusters\n")

seurat_merged <- FindClusters(seurat_merged, resolution = 0.6)
cat("Resolution 0.6 →", length(unique(seurat_merged$seurat_clusters)), "clusters\n")

seurat_merged <- FindClusters(seurat_merged, resolution = 0.8)
cat("Resolution 0.8 →", length(unique(seurat_merged$seurat_clusters)), "clusters\n")


In [ ]:
# 세미나에서는 Resolution 0.4 사용
Idents(seurat_merged) <- "RNA_snn_res.0.4"

DimPlot(seurat_merged, reduction = "umap", label = TRUE, label.size = 4) +
  ggtitle("클러스터 (Resolution 0.4)") +
  theme_minimal(base_family = "nanum")


**▶ 결과 해석**

UMAP 위의 클러스터 번호를 확인하세요:

- **클러스터 수** — 생물학적으로 의미 있는 수? 너무 많거나 적지 않은지 확인
- **클러스터 크기** — 아주 작은 클러스터(< 50 cells)는 artifact일 수 있음
- **클러스터 모양** — 분리가 뚜렷하면 좋음; 퍼져있으면 resolution 조정 고려

> Resolution을 바꾸면서 UMAP을 비교해보세요 (0.2 vs 0.4 vs 0.8)

## Step 11. Marker Gene Identification

각 클러스터를 나머지와 비교하여 **대표 마커 유전자**를 찾습니다.

`FindAllMarkers()` 주요 파라미터:
- `only.pos = TRUE`: **해당 클러스터에서 높게 발현**되는 유전자만
- `min.pct = 0.25`: 최소 **25%** 세포에서 발현
- `logfc.threshold = 0.25`: 최소 **log2FC 0.25** 기준

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_17.png" width="850"/>

*Fig. 14 — 세포 타입 어노테이션 전략: 자동 어노테이션 vs 수동 어노테이션 (Clake ZA et al., Nat Protoc 2019)*

In [ ]:
# 마커 유전자 탐색 (시간 소요: 5~15분)
# only.pos = TRUE: 해당 클러스터에서 높게 발현되는 유전자만
markers <- FindAllMarkers(
  seurat_merged,
  only.pos            = TRUE,
  min.pct             = 0.25,  # 최소 25% 세포에서 발현
  logfc.threshold     = 0.25,  # 최소 log2FC 0.25
  max.cells.per.ident = 200    # 클러스터당 200세포로 다운샘플 (속도·메모리 ↓; 정밀 분석 시 제거)
)

# Top 5 마커 확인
markers %>%
  group_by(cluster) %>%
  slice_max(avg_log2FC, n = 5) %>%
  print(n = Inf)

**▶ 결과 해석**

`FindAllMarkers()` 주요 컬럼:

| 컬럼 | 의미 |
|------|------|
| `avg_log2FC` | 해당 클러스터 vs 나머지의 발현 배수 차이 (log2) |
| `pct.1` | 해당 클러스터에서 발현된 세포 비율 |
| `pct.2` | 나머지 클러스터에서 발현된 세포 비율 |
| `p_val_adj` | BH 보정 p-value |

- **좋은 마커**: 높은 avg_log2FC + 높은 pct.1 + 낮은 pct.2
- 각 클러스터 Top 1~3 유전자를 NCBI/GeneCards에서 검색해보세요

In [ ]:
# Top 10 마커 히트맵
top10 <- markers %>%
  group_by(cluster) %>%
  slice_max(avg_log2FC, n = 10)

DoHeatmap(seurat_merged, features = top10$gene) +
  theme(axis.text.y = element_text(size = 6))

**▶ 결과 해석**

각 클러스터의 Top 마커 유전자 발현 패턴을 확인하세요:

- **진한 색 블록**이 해당 클러스터에서만 나타나면 → 좋은 마커
- 여러 클러스터에 걸쳐 발현 → 범용 마커 (세포 타입 구별에 한계)

> 히트맵 패턴과 논문의 마커 유전자를 비교하여 세포 타입을 추론합니다
> (다음 Step 12 참고)

## ⚠️ 우리 데이터는 "신경망막"이 아니라 "RPE-choroid" 입니다

> **중요:** 이 데이터셋(GSE210543)의 조직은 **RPE(망막색소상피)–맥락막(choroid)** 입니다.
> 논문: *Voigt et al., "Single-cell RNA sequencing reveals transcriptional changes of human choroidal and RPE cells during fetal development, in healthy adult and intermediate AMD", Hum Mol Genet 2023* — **AMD(황반변성)** 연구입니다.
>
> 그래서 광수용체(Rod/Cone), Bipolar, Müller glia 같은 **신경망막 뉴런은 거의 없고**, 대신 **RPE, 맥락막 기질(fibroblast, melanocyte, 내피, 주피, 평활근), 면역·혈액세포**가 대부분입니다.
> QC~클러스터링 파이프라인(Step 0~11)은 **조직과 무관하게 동일**하며, **해석(annotation)만 이 조직에 맞춰** 진행합니다.

### RPE-choroid 마커 키 (논문 기준 — 정답 참고용)

| 세포 타입 | 마커 유전자 |
|----------|------------|
| **RPE** | RPE65, RGR, BEST1, TTR |
| **Choroidal endothelial (CEC)** | VWF, ICAM2, CD34, SOX17 |
| **Melanocytes** | MLANA, PMEL, TYRP1 |
| **Pericytes** | THY1, ITGA1, POSTN |
| **Smooth / Ciliary muscle** | ACTA2, TAGLN, DES, MYH11 |
| **Schwann cells** | PLP1, MPZ, SCN7A |
| **Fibroblasts** | PENK, COL1A1, FBLN1 |
| **Macrophages** | AIF1, C1QA/B/C |
| **T / B / Mast cells** | CD3·PRF1 / CD79A·MS4A1 / KIT·CPA3 |
| **Red blood cells** | HBB, HBA2, HBG |
| **Proliferating** | TOP2A, MKI67 |

> **💡 진행 방식:** FindAllMarkers로 뽑은 각 클러스터의 마커를 위 표와 대조해 세포 타입을 붙입니다 (RPE65→RPE, MLANA→Melanocyte, VWF→CEC, HBB→적혈구 ...).
> ⚠️ **클러스터 번호는 실행마다 달라지므로, 번호가 아니라 "마커"로 판단하세요.**

## Step 12. Cell Type Annotation

### (참고) 신경망막 마커 예시 — Voigt et al. 2022

> ⚠️ 아래 표는 **신경망막(neuroretina) 세포**용 예시입니다. **우리 RPE-choroid 데이터엔 이 세포들이 거의 없어요** — 위 RPE-choroid 마커 키를 사용하세요. (신경망막 분석 시 참고용으로 남겨둡니다.)

| 세포 타입 | 마커 유전자 | 비고 |
|----------|------------|------|
| RGC | SNCG, ISL1, POU4F2 | BRN3 계열 + SNCG |
| Amacrine | TFAP2A, PAX6, GAD1 | — |
| Bipolar | CABP5, PRKCA, GRM6 | VSX2 제외 (Progenitor 혼용 위험) |
| Müller glia | GLUL, RLBP1, SLC1A3 | 3개 모두 고신뢰 마커 |
| Rod | RHO, NRL, RCVRN | NRL = rod master TF |
| Cone | OPN1LW, ARR3, GNGT2 | — |
| Horizontal | LHX1, ONECUT2, PROX1 | — |
| Progenitor | VSX2, FGF19, LIN28B | Young 샘플에 풍부 |
| Microglia | CX3CR1, P2RY12, TMEM119 | — |
| Endothelial | PECAM1, CDH5, VWF | — |

> **💡 Tip:** Bipolar에서 VSX2를 제거했습니다 — 발달기 Progenitor와 혼용 위험이 있어 CABP5/PRKCA로 구별합니다.

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_18.png" width="850"/>

*Fig. 15 — 세포 타입 어노테이션 방법 및 Tip 4: 연구자의 주관이 중요!*

### Task. 어노테이션 테스트 — 3개 세포 타입 (RPE-choroid)

논문에서 **마커 특이성 + 풍부도** 기준으로 선별:

| 세포 타입 | 마커 | 선별 이유 |
|----------|------|---------|
| **RPE** | RPE65, RGR, BEST1 | RPE의 정의 마커 (visual cycle 효소) |
| **Melanocyte** | MLANA, PMEL, TYRP1 | 멜라닌 합성 — 맥락막 색소세포 |
| **Choroidal endothelial** | VWF, ICAM2, CD34 | 맥락막 혈관 내피 |

In [ ]:
# RPE-choroid 마커 유전자 (Voigt et al. 2023, Hum Mol Genet)
markers_RPE    <- c("RPE65", "RGR",   "BEST1")   # 망막색소상피(RPE)
markers_Melano <- c("MLANA", "PMEL",  "TYRP1")   # 멜라닌세포(Melanocyte)
markers_CEC    <- c("VWF",   "ICAM2", "CD34")    # 맥락막 내피(Choroidal endothelial)

# UMAP 위에 유전자 발현 표시 — RPE 마커부터 (신호가 선명하게 뜹니다)
FeaturePlot(seurat_merged, features = markers_RPE, ncol = 3) +
  plot_annotation(title = "RPE 마커 (RPE65 / RGR / BEST1)")

### 어노테이션 검증 — DotPlot

각 클러스터가 어떤 마커를 켜는지 한눈에 봅니다. **큰 점(발현 세포 비율 ↑) + 진한 색(평균 발현 ↑)** 이 겹치는 곳이 그 클러스터의 정체예요. 이걸 보고 아래 `new_labels`를 확정합니다.

In [ ]:
# ── 클러스터 정체 확인: 마커 키를 DotPlot으로 한눈에 ──
Idents(seurat_merged) <- "RNA_snn_res.0.4"   # 어노테이션 기준 resolution 고정

dotplot_markers <- c(
  "RPE65","RGR",           # RPE
  "VWF","SOX17",           # CEC (내피)
  "MLANA","PMEL",          # Melanocyte
  "COX4I2","KCNJ8",        # Pericyte
  "ACTA2","MYH11",         # Smooth muscle
  "MPZ","SCN7A",           # Schwann
  "COL1A1","LUM","PENK",   # Fibroblast
  "AIF1","C1QB",           # Macrophage
  "CD79A","MS4A1",         # B cell
  "CD2","PRF1",            # T/NK
  "KIT","CPA3",            # Mast
  "HBB","RHAG",            # RBC / Erythroid
  "FCGR3B","S100A8",       # Neutrophil
  "KRT12","KRT5",          # Epithelium
  "TOP2A","MKI67",         # Proliferating
  "EYS","NEUROD1",         # Photoreceptor precursor
  "GRIK1","SLC38A8"        # Retinal neuron / Neural prog
)
DotPlot(seurat_merged, features = dotplot_markers, group.by = "RNA_snn_res.0.4") +
  RotatedAxis() +
  ggtitle("클러스터별 마커 발현 (DotPlot)")

In [ ]:
# ── 클러스터 번호 → 세포 타입 매핑 (res 0.4, 23개) ──
# ⚠️ 클러스터 번호는 실행마다 바뀔 수 있어요 — 반드시 마커로 판단하세요!
#    (아래는 이번 분석 run 기준. DotPlot으로 확인 후 수정)
new_labels <- c(
  "0"="Fibroblast","1"="Fibroblast","2"="Retinal_neuron","3"="Melanocyte",
  "4"="RBC","5"="Fibroblast","6"="Neural_prog","7"="CEC",
  "8"="Schwann","9"="Photoreceptor_prec","10"="Macrophage","11"="Pericyte",
  "12"="Proliferating","13"="RPE","14"="Smooth_muscle","15"="T_NK",
  "16"="Macrophage","17"="Glia","18"="Epithelium","19"="Neutrophil",
  "20"="B_cell","21"="Mast_cell","22"="Erythroid"
)  # 17: ETNPPL 기반 tentative (확신 낮음)

# 세포별 클러스터를 순수 문자열로 (res 0.4 기준)
clust <- as.character(seurat_merged$RNA_snn_res.0.4)

# 매핑 — unname()으로 이름 제거! (안 하면 Seurat이 클러스터번호를 barcode로 오인해 에러)
lab <- unname(new_labels[clust])
lab[is.na(lab)] <- paste0("cluster_", clust[is.na(lab)])   # 매핑 안 된 건 cluster_N
seurat_merged$cell_type <- lab

# 최종 UMAP
DimPlot(seurat_merged, reduction = "umap", group.by = "cell_type",
        label = TRUE, label.size = 3, repel = TRUE) +
  ggtitle("Cell Type Annotation (RPE-choroid)") +
  theme_minimal(base_family = "nanum")

In [ ]:
# Young vs Old 세포 타입 구성 비교
prop_table <- table(seurat_merged$group, seurat_merged$cell_type)
prop_df    <- as.data.frame(prop.table(prop_table, margin = 1))
colnames(prop_df) <- c("group", "cell_type", "proportion")

ggplot(prop_df, aes(x = group, y = proportion, fill = cell_type)) +
  geom_bar(stat = "identity") +
  labs(title = "Young vs Old: 세포 타입 구성",
       x = "그룹", y = "비율") +
  theme_minimal(base_family = "nanum")


In [ ]:
# 최종 저장
saveRDS(seurat_merged, file = file.path(SAVE_DIR, "02_seurat_annotated.rds"))
cat("Saved: 02_seurat_annotated.rds\n")

# 요약
cat("\n=== Final Summary ===\n")
cat("Total cells:", ncol(seurat_merged), "\n")
cat("Cell types:\n")
print(table(seurat_merged$cell_type, seurat_merged$group))

---

## 세미나 완료

| 단계 | 완료 내용 |
|------|---------|
| Step 0 | 환경 설정 (Seurat, Harmony) |
| Step 1 | 10X 데이터 로드 (4개 샘플) |
| Step 2-3 | QC 시각화 + 필터링 |
| Step 4-5 | LogNormalize + HVG |
| Step 6 | Cell Cycle (optional) |
| Step 7 | Scaling + PCA |
| Step 8 | Harmony 통합 |
| Step 9 | UMAP |
| Step 10 | Clustering |
| Step 11 | FindAllMarkers |
| Step 12 | Cell Type Annotation |